# Stage 4 -- Compiler-era and temporal shift (E3, E4 / RQ3)
Re-runs the deep suite plus the three strongest baselines under S2 (train v0.4-v0.6,
test v0.7-v0.8; CompilerVersion excluded from inputs) and S3 (train <=2021, test >=2022).
**Accelerator: GPU.** **Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage4_generalization')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

2026-09-25 09:25:43,952 INFO stage stage4_generalization initialised at /kaggle/working/stage4_generalization
2026-09-25 09:25:43,954 INFO seeds=(0, 1, 2) quick=False


In [2]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

2026-09-25 09:25:46,165 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-25 09:25:46,166 INFO family code_metadata        6 features
2026-09-25 09:25:46,167 INFO family interface_source    46 features
2026-09-25 09:25:46,169 INFO family creation_opcodes   168 features
2026-09-25 09:25:46,169 INFO family tx_metadata         10 features
2026-09-25 09:25:46,170 INFO family temporal_position    8 features
2026-09-25 09:25:46,172 INFO family runtime_opcodes    158 features
2026-09-25 09:25:46,173 INFO view F_pre  220 features
2026-09-25 09:25:46,174 INFO view F_post 176 features
2026-09-25 09:25:46,175 INFO view F_all  396 features


In [3]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
# S2 excludes the split variable itself from the inputs
fam_s2 = {k: [c for c in v if c != 'CompilerVersion'] for k, v in fam.items()}

In [4]:
rows = []
label_counts = []
preds = {}   # for the bootstrap significance test (mirrors Stage 3's stage3_test_predictions.npz)
res_path = f'{root}/results/stage4_generalization_results.csv'
done = set()
if os.path.exists(res_path):                 # resume after a kernel restart
    done = {tuple(r) for r in pd.read_csv(res_path)[['model', 'view', 'split', 'seed']].values}
    log.info('resuming: %d runs already checkpointed', len(done))

# extended per R1#1 / R2#7: LAT-mask and WIDENNET now run under S2/S3 too, not S1 only
# extended per the "all classical baselines" footnote: RAkEL and ML-kNN added to the
# shift protocols so all five classical baselines (not just three) are actually evaluated
MODELS = ['LGC', 'LAT', 'LAT-mask', 'WIDENNET', 'BR-trunk',
          'RF-BR', 'LGBM-BR', 'CC-ens', 'RAkEL', 'ML-kNN']
# only these need per-instance predictions saved, to mirror Stage 3's bootstrap test on S2/S3
SAVE_PREDS = {'LGC', 'LAT', 'LAT-mask', 'BR-trunk'}

for sp_name, fam_use in [('S2', fam_s2), ('S3', fam)]:
    for view in scvd.VIEWS:
        d = scvd.make_data(pre, post, lab, fam_use, view, *splits[sp_name])
        M = scvd.cooccurrence(d['y_tr'])

        # R2#6: per-label positive counts for this shifted test split, saved once
        # per (split, view) so rare-class support sits next to the macro-F1 table
        n_pos = d['y_te'].sum(0)
        label_counts.append({'split': sp_name, 'view': view, 'n_test': len(d['y_te']),
                              **{f'n_pos_{scvd.LABEL_SHORT[j]}': int(n_pos[j]) for j in range(8)}})

        op_cols = [c for n in scvd.view_families(fam_use, view) if 'opcodes' in n for c in fam_use[n]]
        n_op = len(op_cols)

        for name in MODELS:
            for seed in SEEDS:
                if (name, view, sp_name, seed) in done:
                    log.info('skip %s F_%s %s seed%d (checkpointed)', name, view, sp_name, seed)
                    continue
                scvd.set_seed(seed)   # fix: deep models below were never actually seeded before
                if name in ('RF-BR', 'LGBM-BR', 'CC-ens', 'RAkEL', 'ML-kNN'):
                    proba = scvd.fit_classical(name, d['X_tr'], d['y_tr'], seed)
                    th = scvd.tune_thresholds(d['y_va'], proba(d['X_va']))
                    pt = proba(d['X_te'])
                elif name == 'WIDENNET':
                    Xop_tr = np.hstack([d['x_tr'][n] for n in d['x_tr'] if 'opcodes' in n])
                    Xop_va = np.hstack([d['x_va'][n] for n in d['x_va'] if 'opcodes' in n])
                    Xop_te = np.hstack([d['x_te'][n] for n in d['x_te'] if 'opcodes' in n])
                    duse = {'x_tr': Xop_tr, 'x_va': Xop_va, 'x_te': Xop_te,
                            'y_tr': d['y_tr'], 'y_va': d['y_va'], 'y_te': d['y_te']}
                    cfg = {'lr': 1e-3, 'dm': 128, 'batch': 256,
                           'epochs': 60 if scvd.quick() else 120}
                    mdl = scvd.WIDENNET(n_op, dm=128)
                    pt, _h = scvd.train_torch(mdl, duse, cfg, log, 'op')
                    pv = scvd.predict(mdl, duse, 'va', 'op')
                    th = scvd.tune_thresholds(d['y_va'], pv)
                else:
                    cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'tau': 0.2, 'batch': 256,
                           'epochs': 60 if scvd.quick() else 120}
                    mdl = (scvd.LGC(d['fams'], M, dm=128, tau=0.2) if name == 'LGC'
                           else scvd.LAT(d['fams'], dm=128, layers=3) if name == 'LAT'
                           else scvd.LAT(d['fams'], dm=128, layers=3, mask_ll=True) if name == 'LAT-mask'
                           else scvd.LAT(d['fams'], dm=128, layers=3, trunk_only=True))
                    pt, _h = scvd.train_torch(mdl, d, cfg, log)
                    pv = scvd.predict(mdl, d, 'va')
                    th = scvd.tune_thresholds(d['y_va'], pv)
                m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
                rows.append({'model': name, 'view': view, 'split': sp_name, 'seed': seed, **m})
                scvd.append_results(root, 'stage4_generalization', [rows[-1]])
                if name in SAVE_PREDS:
                    preds[(name, view, sp_name, seed)] = (d['ids_te'], pt, th)
                log.info('%s %-9s F_%-4s seed%d macro_f1=%.4f', sp_name, name, view, seed, m['macro_f1'])

pd.DataFrame(label_counts).to_csv(f'{root}/results/stage4_label_counts.csv', index=False)

# R1#4 / R1#2 / R2#5: per-instance predictions + thresholds, so the same bootstrap
# significance test Stage 3 runs on S1 can also run on S2/S3 (mirrors stage3_test_predictions.npz)
np.savez_compressed(f'{root}/results/stage4_test_predictions.npz',
                    **{f'{n}__{v}__{sp}__{s}__ids': i for (n, v, sp, s), (i, _p, _t) in preds.items()},
                    **{f'{n}__{v}__{sp}__{s}__probs': p for (n, v, sp, s), (_i, p, _t) in preds.items()})
json.dump({f'{n}|{v}|{sp}|{s}': list(map(float, t)) for (n, v, sp, s), (_i, _p, t) in preds.items()},
          open(f'{root}/results/stage4_thresholds.json', 'w'))


2026-09-25 09:25:54,929 INFO   ep 000 loss 2.1136 val_macro_f1 0.2927 *
2026-09-25 09:25:55,085 INFO   ep 001 loss 1.1550 val_macro_f1 0.2387
2026-09-25 09:25:55,239 INFO   ep 002 loss 1.4857 val_macro_f1 0.0781
2026-09-25 09:25:55,401 INFO   ep 003 loss 1.4363 val_macro_f1 0.3330 *
2026-09-25 09:25:55,578 INFO   ep 004 loss 1.0815 val_macro_f1 0.3671 *
2026-09-25 09:25:55,749 INFO   ep 005 loss 1.1810 val_macro_f1 0.4208 *
2026-09-25 09:25:55,911 INFO   ep 006 loss 0.9554 val_macro_f1 0.3709
2026-09-25 09:25:56,075 INFO   ep 007 loss 1.1846 val_macro_f1 0.4995 *
2026-09-25 09:25:56,244 INFO   ep 008 loss 0.9284 val_macro_f1 0.5276 *
2026-09-25 09:25:56,400 INFO   ep 009 loss 1.2010 val_macro_f1 0.4309
2026-09-25 09:25:56,564 INFO   ep 010 loss 1.1014 val_macro_f1 0.5867 *
2026-09-25 09:25:56,737 INFO   ep 011 loss 0.8451 val_macro_f1 0.4927
2026-09-25 09:25:56,938 INFO   ep 012 loss 0.8980 val_macro_f1 0.4262
2026-09-25 09:25:57,115 INFO   ep 013 loss 1.0935 val_macro_f1 0.6092 *
2026

In [18]:
# R1#2 / R1#4 / R2#5: same paired-bootstrap significance test the Stage 5 notebook
# runs on S1, now run on S2 and S3 using stage4_test_predictions.npz / stage4_thresholds.json.
# Self-contained (doesn't assume Stage 3/5's mf1/boot are already defined in this session).
def _resolve_stage4_results():
    """Don't trust the ambient `root` -- it gets overwritten by whichever stage's
    scvd.setup_stage(...) ran most recently in this kernel. Look for the files
    directly: first in this session's own Stage 4 working dir, then in an
    attached Stage 4 input dataset (if this cell is run from a different
    session/notebook than the one that trained Stage 4)."""
    local = '/kaggle/working/stage4_generalization/results'
    if os.path.exists(os.path.join(local, 'stage4_test_predictions.npz')):
        return local
    hit = scvd._find_input({'npz': 'stage4_test_predictions.npz', 'th': 'stage4_thresholds.json'})
    if hit and 'npz' in hit and 'th' in hit:
        return os.path.dirname(hit['npz'])
    raise FileNotFoundError(
        "Could not find stage4_test_predictions.npz in this session's working dir or under "
        "/kaggle/input. If running this in a different session than Stage 4's training loop, "
        "attach the pushed 'scvd-stage4-generalization' dataset as input first.")

res4 = _resolve_stage4_results()
P4 = np.load(os.path.join(res4, 'stage4_test_predictions.npz'))
TH4 = json.load(open(os.path.join(res4, 'stage4_thresholds.json')))

def mf1(pred, yy):
    tp = ((pred == 1) & (yy == 1)).sum(0); fp = ((pred == 1) & (yy == 0)).sum(0)
    fn = ((pred == 0) & (yy == 1)).sum(0)
    return float(np.mean(2*tp / np.clip(2*tp + fp + fn, 1e-12, None)))

def boot4(pa, pb, yy, n=10000, seed=0):
    rs = np.random.RandomState(seed); N = len(yy); d = np.empty(n)
    for t in range(n):
        ix = rs.randint(0, N, N)
        d[t] = mf1(pa[ix], yy[ix]) - mf1(pb[ix], yy[ix])
    return np.percentile(d, 2.5), np.percentile(d, 97.5)

def thp4(name, view, sp, s):
    return (P4[f'{name}__{view}__{sp}__{s}__probs'] >= np.asarray(TH4[f'{name}|{view}|{sp}|{s}'])).astype(int)

for sp in ('S2', 'S3'):
    ids4 = P4[f'LGC__pre__{sp}__0__ids']
    y4 = lab.set_index('contractID').loc[ids4, scvd.LABELS].to_numpy(int)
    for view in ('pre', 'post', 'all'):
        for a, b in [('LGC', 'BR-trunk'), ('LAT', 'BR-trunk'), ('LAT', 'LAT-mask')]:
            msgs = []
            for s in (0, 1, 2):
                pa, pb = thp4(a, view, sp, s), thp4(b, view, sp, s)
                lo, hi = boot4(pa, pb, y4, 10000, s)
                d_pt = mf1(pa, y4) - mf1(pb, y4)
                msgs.append(f"s{s}: d={d_pt:+.4f} [{lo:+.4f},{hi:+.4f}]")
            print(f'{sp} {view:4s} {a:8s} vs {b:8s} | ' + ' | '.join(msgs))

S2 pre  LGC      vs BR-trunk | s0: d=-0.0302 [-0.0357,-0.0241] | s1: d=+0.0036 [-0.0025,+0.0099] | s2: d=-0.0223 [-0.0274,-0.0177]
S2 pre  LAT      vs BR-trunk | s0: d=-0.0034 [-0.0080,+0.0011] | s1: d=-0.0102 [-0.0159,-0.0047] | s2: d=+0.0137 [+0.0072,+0.0200]
S2 pre  LAT      vs LAT-mask | s0: d=-0.0199 [-0.0248,-0.0152] | s1: d=-0.0176 [-0.0221,-0.0131] | s2: d=+0.0077 [+0.0033,+0.0123]
S2 post LGC      vs BR-trunk | s0: d=+0.0013 [-0.0050,+0.0079] | s1: d=-0.0156 [-0.0215,-0.0099] | s2: d=-0.0119 [-0.0159,-0.0078]
S2 post LAT      vs BR-trunk | s0: d=+0.0057 [-0.0004,+0.0119] | s1: d=-0.0162 [-0.0220,-0.0106] | s2: d=-0.0017 [-0.0064,+0.0032]
S2 post LAT      vs LAT-mask | s0: d=-0.0109 [-0.0167,-0.0053] | s1: d=+0.0017 [-0.0034,+0.0066] | s2: d=+0.0166 [+0.0115,+0.0216]
S2 all  LGC      vs BR-trunk | s0: d=+0.0023 [-0.0024,+0.0072] | s1: d=-0.0135 [-0.0186,-0.0085] | s2: d=-0.0224 [-0.0277,-0.0172]
S2 all  LAT      vs BR-trunk | s0: d=+0.0017 [-0.0038,+0.0078] | s1: d=+0.0016 [-0.

In [6]:
summ, sp = scvd.write_summary(root, 'stage4_generalization')
s1 = scvd._find_input({'s3': 'stage3_deep_summary.csv'})   # was 'stage3_summary.csv' -- never matched, S1 silently never merged
if s1:
    summ = pd.concat([pd.read_csv(s1['s3']), summ], ignore_index=True)
scvd.plot_shift_degradation(summ, root, log)
scvd.tex_table(summ[summ['view'] == 'all'], f'{root}/tables/stage4_shift.tex',
               ['macro_f1', 'micro_f1'], 'Macro-F1 under S1/S2/S3 on F_all.', 'tab:stage4shift')
summ.round(4)

2026-09-25 12:52:33,020 INFO maxp pruned
2026-09-25 12:52:33,033 INFO cmap pruned
2026-09-25 12:52:33,040 INFO kern dropped
2026-09-25 12:52:33,041 INFO post pruned
2026-09-25 12:52:33,044 INFO FFTM dropped
2026-09-25 12:52:33,045 INFO GPOS pruned
2026-09-25 12:52:33,047 INFO GSUB pruned
2026-09-25 12:52:33,053 INFO glyf pruned
2026-09-25 12:52:33,054 INFO Added gid0 to subset
2026-09-25 12:52:33,055 INFO Added first four glyphs to subset
2026-09-25 12:52:33,055 INFO Closing glyph list over 'MATH': 41 glyphs before
2026-09-25 12:52:33,056 INFO Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'L', 'M', 'N', 'R', 'S', 'T', 'W', 'a', 'c', 'e', 'eight', 'four', 'hyphen', 'k', 'm', 'n', 'nonmarkingreturn', 'o', 'one', 'parenleft', 'parenright', 'period', 'r', 's', 'six', 'space', 't', 'three', 'two', 'u', 'zero']
2026-09-25 12:52:33,058 INFO Glyph IDs:   [0, 1, 2, 3, 11, 12, 16, 17, 19, 20, 21, 22, 23, 25, 27, 36, 37, 38, 39, 40, 41, 42, 44, 47, 48, 49, 53, 54, 55, 

,model,view,split,seed,lr,dm,layers,epochs,batch,tau,...,f1_FrontR_sd,f1_TimeMan_sd,auc_Reent_sd,auc_AccCtrl_sd,auc_Arith_sd,auc_UnchkRet_sd,auc_DoS_sd,auc_BadRnd_sd,auc_FrontR_sd,auc_TimeMan_sd
0,BR-trunk,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0102,0.0083,0.0004,0.0011,0.0021,0.0024,0.0048,0.0054,0.0137,0.0022
1,BR-trunk,post,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.3,...,0.0430,0.0086,0.0040,0.0202,0.0055,0.0049,0.0080,0.0177,0.0135,0.0027
2,BR-trunk,pre,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0565,0.0077,0.0052,0.0030,0.0026,0.0017,0.0066,0.0015,0.0077,0.0016
3,LAT,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0322,0.0090,0.0011,0.0007,0.0040,0.0028,0.0044,0.0133,0.0075,0.0016
4,LAT,post,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.1,...,0.0216,0.0085,0.0060,0.0021,0.0117,0.0038,0.0043,0.0061,0.0108,0.0029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,WIDENNET,all,S3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.0118,0.0108,0.0082,0.0418,0.0046,0.0037,0.0033,0.0616,0.0090
71,WIDENNET,post,S2,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.0110,0.0189,0.0208,0.0261,0.0623,0.0465,0.0160,0.0884,0.0040
72,WIDENNET,post,S3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.0030,0.0113,0.0074,0.0342,0.0462,0.0108,0.0449,0.0640,0.0168
73,WIDENNET,pre,S2,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.0095,0.0128,0.0163,0.0182,0.0616,0.0226,0.0567,0.0824,0.0400


In [7]:
title = 'SCVD Stage 4: compiler-era and temporal shift'  # 46 chars, under the cap
url = scvd.push_dataset(root, 'stage4_generalization', 'scvd-stage4-generalization', title, log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)

Starting upload for file results.zip


100%|██████████| 25.1M/25.1M [00:00<00:00, 60.2MB/s]


Upload successful: results.zip (25MB)
Starting upload for file plots.zip


100%|██████████| 78.6k/78.6k [00:00<00:00, 413kB/s]


Upload successful: plots.zip (79KB)
Starting upload for file logs.zip


100%|██████████| 139k/139k [00:00<00:00, 750kB/s]


Upload successful: logs.zip (139KB)
Starting upload for file tables.zip


100%|██████████| 555/555 [00:00<00:00, 3.12kB/s]


Upload successful: tables.zip (555B)
2026-09-25 12:52:37,902 INFO pushed new version -> https://www.kaggle.com/datasets/smnihalahmed/scvd-stage4-generalization
DATASET: https://www.kaggle.com/datasets/smnihalahmed/scvd-stage4-generalization


In [7]:
import json, numpy as np, pandas as pd

STAGE3 = '/kaggle/input/datasets/smnihalahmed/scvd-stage3-deep'
P = np.load(f'{STAGE3}/results/stage3_test_predictions.npz')
TH = json.load(open(f'{STAGE3}/results/stage3_thresholds.json'))
ids = P['LGC__pre__0__ids']
y = lab.set_index('contractID').loc[ids, scvd.LABELS].to_numpy(int)

def mf1(pred, yy):
    tp = ((pred == 1) & (yy == 1)).sum(0); fp = ((pred == 1) & (yy == 0)).sum(0)
    fn = ((pred == 0) & (yy == 1)).sum(0)
    return float(np.mean(2*tp / np.clip(2*tp + fp + fn, 1e-12, None)))

def boot(pa, pb, n=10000, seed=0):
    rs = np.random.RandomState(seed); N = len(y); d = np.empty(n)
    for t in range(n):
        ix = rs.randint(0, N, N)
        d[t] = mf1(pa[ix], y[ix]) - mf1(pb[ix], y[ix])
    return np.percentile(d, 2.5), np.percentile(d, 97.5)

def thp(name, view, s):
    return (P[f'{name}__{view}__{s}__probs'] >= np.asarray(TH[f'{name}|{view}|{s}'])).astype(int)

for view in ('pre', 'post', 'all'):
    for a, b in [('LGC','BR-trunk'), ('LAT','BR-trunk'), ('LAT','LAT-mask')]:
        msgs = []
        for s in (0, 1, 2):
            pa, pb = thp(a, view, s), thp(b, view, s)
            lo, hi = boot(pa, pb, 10000, s)
            msgs.append(f"s{s}: d={mf1(pa, y)-mf1(pb, y):+.4f} [{lo:+.4f},{hi:+.4f}]")
        print(f'{view:4s} {a:8s} vs {b:8s} | ' + ' | '.join(msgs))

for a, b in [('LGC','BR-trunk'), ('LAT','LAT-mask')]:
    pa = (np.mean([P[f'{a}__all__{s}__probs'] for s in (0,1,2)], 0)
          >= np.mean([TH[f'{a}|all|{s}'] for s in (0,1,2)], 0)).astype(int)
    pb = (np.mean([P[f'{b}__all__{s}__probs'] for s in (0,1,2)], 0)
          >= np.mean([TH[f'{b}|all|{s}'] for s in (0,1,2)], 0)).astype(int)
    rs = np.random.RandomState(0); N = len(y); ps = np.empty(8)
    for j in range(8):
        d = np.empty(4000)
        for t in range(4000):
            ix = rs.randint(0, N, N)
            fa = 2*((pa[ix,j]==1)&(y[ix,j]==1)).sum()
            fb = 2*((pb[ix,j]==1)&(y[ix,j]==1)).sum()
            d[t] = (fa/max(1e-12, fa+((pa[ix,j]==1)&(y[ix,j]==0)).sum()+((pa[ix,j]==0)&(y[ix,j]==1)).sum())
                  - fb/max(1e-12, fb+((pb[ix,j]==1)&(y[ix,j]==0)).sum()+((pb[ix,j]==0)&(y[ix,j]==1)).sum()))
        ps[j] = (d <= 0).mean() if d.mean() > 0 else (d >= 0).mean()
    o = np.argsort(ps)
    ph = np.minimum.accumulate(ps[o] * (8 - np.arange(8)))[np.argsort(o)]
    print(f'Holm {a} vs {b}:', ' '.join(f'{scvd.LABEL_SHORT[j]}={min(ph[j],1.0):.4f}' for j in range(8)))

pre  LGC      vs BR-trunk | s0: d=-0.0050 [-0.0181,+0.0084] | s1: d=-0.0108 [-0.0262,+0.0045] | s2: d=+0.0029 [-0.0119,+0.0181]
pre  LAT      vs BR-trunk | s0: d=-0.0051 [-0.0184,+0.0080] | s1: d=-0.0072 [-0.0214,+0.0064] | s2: d=-0.0017 [-0.0154,+0.0120]
pre  LAT      vs LAT-mask | s0: d=+0.0014 [-0.0131,+0.0154] | s1: d=-0.0156 [-0.0311,-0.0005] | s2: d=+0.0019 [-0.0126,+0.0167]
post LGC      vs BR-trunk | s0: d=-0.0400 [-0.0558,-0.0248] | s1: d=-0.0053 [-0.0211,+0.0110] | s2: d=+0.0042 [-0.0113,+0.0197]
post LAT      vs BR-trunk | s0: d=+0.0071 [-0.0070,+0.0211] | s1: d=+0.0138 [-0.0004,+0.0282] | s2: d=+0.0236 [+0.0085,+0.0392]
post LAT      vs LAT-mask | s0: d=+0.0110 [-0.0041,+0.0260] | s1: d=+0.0142 [-0.0006,+0.0292] | s2: d=+0.0088 [-0.0041,+0.0221]
all  LGC      vs BR-trunk | s0: d=-0.0216 [-0.0342,-0.0092] | s1: d=-0.0046 [-0.0190,+0.0095] | s2: d=-0.0182 [-0.0310,-0.0055]
all  LAT      vs BR-trunk | s0: d=-0.0059 [-0.0175,+0.0057] | s1: d=-0.0023 [-0.0164,+0.0117] | s2: d=-0